# Data preprocessing and initial classification

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

In [ ]:
heart_disease = fetch_ucirepo(id=45)

X: pd.DataFrame = heart_disease.data.features
y_5_classes: pd.Series = heart_disease.data.targets['num']

In [ ]:
mask = ~np.asarray(pd.isna(X).any(axis=1))

X = X[mask]
y_5_classes = y_5_classes[mask]

In [ ]:
y_binary = (y_5_classes > 0).astype(int)

In [ ]:
TO_ONE_HOT_ENCODE = ["cp", "restecg", "thal", "slope"]
TO_TRANSFORM = ["trestbps", "chol", "thalach", "oldpeak"]
NUM_COLS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler


def build_pipeline(model_name, onehot, power, sampler_name):
    cont = PowerTransformer(method="yeo-johnson") if power else "passthrough"
    cat = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if onehot else "passthrough"

    standard = StandardScaler()

    prep = ColumnTransformer(
        transformers=[
            ("cont", cont, TO_TRANSFORM),
            ('stand', standard, NUM_COLS),
            ("cat", cat, TO_ONE_HOT_ENCODE),
        ],
        remainder="passthrough",
    )

    samplers = {
        "none": "passthrough",
        "random": RandomOverSampler(random_state=42),
        "smote": SMOTE(random_state=42),
    }
    models = {
        "gnb": GaussianNB(),
        "rf": RandomForestClassifier(random_state=42),
    }
    return Pipeline([
        ("prep", prep),
        ("sampler", samplers[sampler_name]),
        ("clf", models[model_name]),
    ])


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

scoring = ["accuracy","f1_macro"]

## y with 5 classes

In [ ]:
import itertools
from pprint import pprint

grid = itertools.product (
    ['gnb', 'rf'],
    [True, False],
    [True, False],
    ['none', 'random', 'smote']
)

for model, ohe, transformed, oversampler in grid:
    print(f'Model: {model} | Ohe: {ohe} | Transformed: {transformed} | Oversampler: {oversampler}')
    
    pipe = build_pipeline(model, ohe, transformed, oversampler)
    res = cross_validate(pipe, X, y_5_classes, cv=cv, scoring=scoring)

    metric_results = {}

    for metric in scoring:
        metric_results[metric] = float(res[f"test_{metric}"].mean())

    pprint(metric_results)
    print()

## y with 2 classes

In [ ]:
import itertools
from pprint import pprint

grid_2 = itertools.product (
    ['gnb', 'rf'],
    [True, False],
    [True, False],
    ['none']
)

for model, ohe, transformed, oversampler in grid_2:
    print(f'Model: {model} | Ohe: {ohe} | Transformed: {transformed}')
    
    pipe = build_pipeline(model, ohe, transformed, oversampler)
    res = cross_validate(pipe, X, y_binary, cv=cv, scoring=scoring)

    metric_results = {}

    for metric in scoring:
        metric_results[metric] = float(res[f"test_{metric}"].mean())

    pprint(metric_results)
    print()